---
# ===============================================================
# PHASE 4 — Transportation LP
# ===============================================================

**Objective:** Minimize total monthly shipping cost subject to supply and demand constraints.

**Decision variables:** x[mill][co-op] = units shipped from that mill to that co-op

**Scenarios:**
1. **Baseline** — all three mills operating
2. **Wichita Offline** — quantify the annual freight value of the Wichita mill (Rachel's board question)
3. **Integration Insight** — LP cost under forecast-based demand vs. historical-average demand

In [ ]:
# Build co-op demand vector for Month 25
# Step 1: total M25 forecast across all categories
m25_total = sum(winners[cat]['M25Forecast'] for cat in categories)
print(f'Total M25 forecast (all categories, all co-ops): {m25_total:,.1f} units')

# Step 2: each co-op's historical share of total volume
coop_share = sales.groupby('CoOp')['UnitsSold'].sum() / sales['UnitsSold'].sum()
print('\nHistorical co-op share:')
print(coop_share.round(4))

# Step 3: allocate M25 total to each co-op
demand_m25 = (coop_share * m25_total).round(0).astype(int).to_dict()
print('\nMonth-25 demand by co-op:')
for c, v in demand_m25.items():
    print(f'  {c}: {v:,}')
print(f'Total: {sum(demand_m25.values()):,}')

Total M25 forecast (all categories, all co-ops): 5,019.2 units

Historical co-op share:
CoOp
Des Moines IA    0.21
Lincoln NE       0.19
Springfield MO   0.20
Topeka KS        0.22
Tulsa OK         0.17
Name: UnitsSold, dtype: float64

Month-25 demand by co-op:
  Des Moines IA: 1,070
  Lincoln NE: 952
  Springfield MO: 1,001
  Topeka KS: 1,119
  Tulsa OK: 878
Total: 5,020


In [ ]:
# LP inputs
MILLS_LP = mills['Mill'].tolist()          # ['Topeka Mill', 'Wichita Mill', 'Omaha Mill']
COOPS_LP = list(demand_m25.keys())
capacity = mills.set_index('Mill')['MonthlyCapacity'].to_dict()

# Cost matrix: index=Origin (mill name), columns=Destination (co-op name)
cost_matrix = shipping.pivot(index='Origin', columns='Destination', values='CostPerUnit')

print('Mill capacities:', capacity)
print('\nShipping cost matrix ($/unit):')
cost_matrix

Mill capacities: {'Topeka Mill': 2800, 'Wichita Mill': 1700, 'Omaha Mill': 2600}

Shipping cost matrix ($/unit):


Destination,Des Moines IA,Lincoln NE,Springfield MO,Topeka KS,Tulsa OK
Origin,,,,,
Omaha Mill,4,3,12,9,13
Topeka Mill,10,8,6,3,7
Wichita Mill,14,11,7,5,4


In [ ]:
def solve_transport_lp(demand_dict, cap_dict, label=''):
    """
    Solve the Heartland transportation LP.
    demand_dict : {co-op: units needed this month}
    cap_dict    : {mill: monthly capacity}
    Returns     : (prob, shipment_df, total_cost)
    """
    prob = pulp.LpProblem(f'Heartland_{label}', pulp.LpMinimize)

    # Decision variables: x[mill][coop] = units shipped
    x = {m: {c: pulp.LpVariable(f'x_{m[:3]}_{c[:3]}', lowBound=0)
              for c in COOPS_LP}
         for m in MILLS_LP}

    # Objective: minimize total shipping cost
    prob += pulp.lpSum(
        cost_matrix.loc[m, c] * x[m][c]
        for m in MILLS_LP
        for c in COOPS_LP
    )

    # Supply constraints (each mill ≤ its monthly capacity)
    for m in MILLS_LP:
        prob += (
            pulp.lpSum(x[m][c] for c in COOPS_LP) <= cap_dict.get(m, 0),
            f'Supply_{m}'
        )

    # Demand constraints (each co-op must receive exactly its demand)
    for c in COOPS_LP:
        prob += (
            pulp.lpSum(x[m][c] for m in MILLS_LP) == demand_dict[c],
            f'Demand_{c}'
        )

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    # Extract shipment matrix
    shipment = pd.DataFrame(
        {m: {c: round(pulp.value(x[m][c]) or 0, 1) for c in COOPS_LP}
         for m in MILLS_LP}
    ).T

    return prob, shipment, pulp.value(prob.objective)

print('LP solver defined.')

LP solver defined.


In [ ]:
# ── Scenario 1: Baseline — all 3 mills ────────────────────────────────────────
prob_base, ship_base, cost_base = solve_transport_lp(demand_m25, capacity, 'Baseline')

print(f'Baseline LP Status:   {pulp.LpStatus[prob_base.status]}')
print(f'Optimal monthly cost: ${cost_base:,.2f}')
print(f'Annualized cost:      ${cost_base * 12:,.2f}')
print('\nOptimal Shipment Matrix (units):')
print(ship_base.to_string())

Baseline LP Status:   Optimal
Optimal monthly cost: $20,011.00
Annualized cost:      $240,132.00

Optimal Shipment Matrix (units):
              Des Moines IA  Lincoln NE  Springfield MO  Topeka KS  Tulsa OK
Topeka Mill            0.00        0.00        1,001.00   1,119.00      0.00
Wichita Mill           0.00        0.00            0.00       0.00    878.00
Omaha Mill         1,070.00      952.00            0.00       0.00      0.00


In [ ]:
print('=== Mill Utilization & Binding Constraints ===')
for m in MILLS_LP:
    shipped = ship_base.loc[m].sum()
    cap     = capacity[m]
    tag     = 'BINDING' if abs(shipped - cap) < 1 else f'slack = {cap - shipped:.0f}'
    print(f'  {m}: {shipped:,.0f} / {cap:,} ({shipped/cap*100:.1f}%)  [{tag}]')

print('\n=== Demand Satisfaction ===')
for c in COOPS_LP:
    received = ship_base[c].sum()
    needed   = demand_m25[c]
    status   = 'OK' if abs(received - needed) < 1 else 'SHORTFALL'
    print(f'  {c}: {received:.0f} received / {needed} needed  [{status}]')

=== Mill Utilization & Binding Constraints ===
  Topeka Mill: 2,120 / 2,800 (75.7%)  [slack = 680]
  Wichita Mill: 878 / 1,700 (51.6%)  [slack = 822]
  Omaha Mill: 2,022 / 2,600 (77.8%)  [slack = 578]

=== Demand Satisfaction ===
  Des Moines IA: 1070 received / 1070 needed  [OK]
  Lincoln NE: 952 received / 952 needed  [OK]
  Springfield MO: 1001 received / 1001 needed  [OK]
  Topeka KS: 1119 received / 1119 needed  [OK]
  Tulsa OK: 878 received / 878 needed  [OK]


In [ ]:
# ── Scenario 2: Wichita Offline ───────────────────────────────────────────────
cap_no_wichita = {m: (0 if 'Wichita' in m else v) for m, v in capacity.items()}

total_demand  = sum(demand_m25.values())
remaining_cap = sum(cap_no_wichita.values())

print(f'Total M25 demand:             {total_demand:,} units')
print(f'Topeka + Omaha capacity only: {remaining_cap:,} units')

if remaining_cap >= total_demand:
    prob_nw, ship_nw, cost_nw = solve_transport_lp(demand_m25, cap_no_wichita, 'NoWichita')
    w_annual = (cost_nw - cost_base) * 12
    print(f'\nNo-Wichita LP Status:    {pulp.LpStatus[prob_nw.status]}')
    print(f'No-Wichita monthly cost: ${cost_nw:,.2f}')
    print(f'Baseline monthly cost:   ${cost_base:,.2f}')
    print(f'\n*** Annual freight value of keeping Wichita open: ${w_annual:,.2f} ***')
    print('(Closing Wichita would raise annual freight spend by this amount)')
    print('\nNo-Wichita Shipment Matrix:')
    print(ship_nw.to_string())
else:
    print('\nINFEASIBLE — Topeka + Omaha cannot cover demand without Wichita.')
    print('The Wichita mill is essential for supply continuity.')
    cost_nw, w_annual = None, None

Total M25 demand:             5,020 units
Topeka + Omaha capacity only: 5,400 units

No-Wichita LP Status:    Optimal
No-Wichita monthly cost: $23,833.00
Baseline monthly cost:   $20,011.00

*** Annual freight value of keeping Wichita open: $45,864.00 ***
(Closing Wichita would raise annual freight spend by this amount)

No-Wichita Shipment Matrix:
              Des Moines IA  Lincoln NE  Springfield MO  Topeka KS  Tulsa OK
Topeka Mill            0.00        0.00        1,001.00   1,119.00    680.00
Wichita Mill           0.00        0.00            0.00       0.00      0.00
Omaha Mill         1,070.00      952.00            0.00       0.00    198.00


In [ ]:
# ── Integration Insight: Value of Forecasting ─────────────────────────────────
# Run LP with naive historical-average demand instead of Phase 2 forecast demand.
# The dollar difference is the "value of forecasting" for the routing plan.

hist_demand = (sales.groupby('CoOp')['UnitsSold'].sum() / 24).round(0).astype(int).to_dict()

print('Historical average monthly demand per co-op:')
for c, v in hist_demand.items(): print(f'  {c}: {v:,}')
print('\nForecast-based M25 demand per co-op:')
for c, v in demand_m25.items(): print(f'  {c}: {v:,}')

prob_hist, ship_hist, cost_hist = solve_transport_lp(hist_demand, capacity, 'Historical')

vof_annual = abs(cost_hist - cost_base) * 12
print(f'\nLP with historical-average demand: ${cost_hist:,.2f}/month')
print(f'LP with M25 forecast demand:       ${cost_base:,.2f}/month')
print(f'\n*** Value of Forecasting (annual): ${vof_annual:,.2f} ***')

Historical average monthly demand per co-op:
  Des Moines IA: 1,027
  Lincoln NE: 914
  Springfield MO: 961
  Topeka KS: 1,074
  Tulsa OK: 843

Forecast-based M25 demand per co-op:
  Des Moines IA: 1,070
  Lincoln NE: 952
  Springfield MO: 1,001
  Topeka KS: 1,119
  Tulsa OK: 878

LP with historical-average demand: $19,210.00/month
LP with M25 forecast demand:       $20,011.00/month

*** Value of Forecasting (annual): $9,612.00 ***


In [ ]:
print('╔══════════════════════════════════════════════════════════════════╗')
print('║  PHASE 4 SCENARIO SUMMARY — Heartland Grain & Feed             ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print(f'║  Baseline (all 3 mills)     Monthly: ${cost_base:>10,.2f}             ║')
print(f'║                             Annual:  ${cost_base*12:>10,.2f}             ║')
if cost_nw is not None:
    print(f'║  No Wichita                 Monthly: ${cost_nw:>10,.2f}             ║')
    print(f'║                             Annual:  ${cost_nw*12:>10,.2f}             ║')
    print(f'║  → Wichita annual value:    ${w_annual:>10,.2f}             ║')
else:
    print('║  No Wichita: INFEASIBLE — mill is essential for supply          ║')
print(f'║  Historical demand LP       Monthly: ${cost_hist:>10,.2f}             ║')
print(f'║  → Value of Forecasting:    ${vof_annual:>10,.2f} /year         ║')
print('╚══════════════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════════════╗
║  PHASE 4 SCENARIO SUMMARY — Heartland Grain & Feed             ║
╠══════════════════════════════════════════════════════════════════╣
║  Baseline (all 3 mills)     Monthly: $ 20,011.00             ║
║                             Annual:  $240,132.00             ║
║  No Wichita                 Monthly: $ 23,833.00             ║
║                             Annual:  $285,996.00             ║
║  → Wichita annual value:    $ 45,864.00             ║
║  Historical demand LP       Monthly: $ 19,210.00             ║
║  → Value of Forecasting:    $  9,612.00 /year         ║
╚══════════════════════════════════════════════════════════════════╝
